# Random baseline for latent object-composition

**The claim under test.** `delta_h_analysis` §7 (2026-08-03) found that in latent space
`[move obj0] + [move obj1]` recovers most of what `[move both]` does — `cos(composed, direct)`
**+0.873**, relative residual **0.39–0.69**, `‖composed‖/‖direct‖` **1.13** — and called it
*"the strongest positive in the editability thread so far"*. On 2026-08-05 the **decode-space**
version of that claim was found to be an artifact of the affine decoder
(`decode(h₀+d₁+d₂) = decode(h₀+d₁) + decode(h₀+d₂) − decode(h₀)` holds identically for any affine
map, to 6.6e-08). The **state-space** readout was explicitly recorded as *unaffected*, and it is
what this notebook re-examines.

**The question (Sevan, 2026-08-21):** is latent composition a *learned* property, or does a
randomly initialised network do it too?

**Why that is a serious worry and not a formality.** Additivity is a **first-order Taylor property
of any smooth map**:

$$f(x + \delta_A + \delta_B) \;\approx\; f(x) + J\delta_A + J\delta_B \;=\; f(x{+}\delta_A) + f(x{+}\delta_B) - f(x)$$

with error second-order in ‖δ‖. Training is irrelevant to this. Random weights do not give a random
*function* — they give a smooth deterministic one, and at standard initialisation a small input
perturbation rarely flips ReLU signs, so the network acts as a **fixed linear operator** in that
neighbourhood. Random linear maps preserve additivity exactly. So finding additivity is weak
evidence of structure unless it is measured against the right controls.

**Three controls this notebook adds**, each of which a first pass on 2026-08-21 got wrong:

| control | why | what a first pass did |
|---|---|---|
| **real teleport targets** from `edits.h5` | displacing every object the same way makes Δ_A ∥ Δ_B and inflates every null | displaced all objects toward the frustum centre by 1.0 in x |
| **triviality baseline** `cos(Δ_A, Δ_AB)` | `Δ_AB` contains the A displacement *by construction*, so the composed cosine can be high without composition doing work | omitted |
| **shuffle BOTH deltas** | shuffling one leaves Δ_A's overlap with Δ_AB intact, so the "floor" is bounded below by the triviality baseline | shuffled one, and reported a +0.38 "floor" that was meaningless |
| **displacement-magnitude sweep** | separates "additive because locally linear" from "additive because object-factored" | omitted |

⚠ **What cannot be measured here.** §7's *decisive* readout was the composed state **applied and
rolled out**, scored as % of the direct edit's Edit-Index gain (83–87%) — it warned in those words
that *"vector agreement alone can mislead"*. A randomly initialised decoder emits garbage, so that
readout is undefined for the random arms and **only the vector metrics transfer**. Every number
below is a vector metric, and must be quoted as such.


## Definitions — every term and metric

Let `h_X` be the flat latent after teacher-forcing the rendered history of world `X` up to the edit
frame, and `Δ_X = h_X − h_base`. Four worlds per episode:

| world | object 0 | object 1 |
|---|---|---|
| `base` | true trajectory | true trajectory |
| `A` | travels to its teleport target | true trajectory |
| `B` | true trajectory | travels to its teleport target |
| `AB` | travels to its target | travels to its target |

A moved object follows a **constant-velocity line that arrives at its target at the edit frame**,
and the whole history is re-rendered — the construction in `edit_directions.py`, i.e. the
**counterfactual overwrite** mechanism from `METRICS_AND_EDITORS.md`. It is memoryless, so it
isolates the *configuration → latent* map.

| metric | formula | units | better | notes |
|---|---|---|---|---|
| **composed cosine** | `cos(Δ_A + Δ_B, Δ_AB)` | −1…1 | ↑ | the headline of §7; **scale-blind** |
| **triviality baseline** | `cos(Δ_A, Δ_AB)` | −1…1 | — | *not* a target. How much of the composed cosine is mere overlap, since `Δ_AB` contains the A displacement. The composed cosine only means something to the extent it **exceeds** this |
| **relative residual** | `‖Δ_AB − (Δ_A + Δ_B)‖ / ‖Δ_AB‖` | ratio | ↓ | the fraction of the true double-edit displacement that composition fails to explain. **0** = perfect; **1** = as wrong as predicting nothing. Catches what cosine cannot: a prediction that points correctly but is half the length reads cosine 1.000 and residual 0.5 |
| **residual, one delta** | `‖Δ_AB − Δ_A‖ / ‖Δ_AB‖` | ratio | — | the residual's triviality baseline |
| **norm ratio** | `‖Δ_A + Δ_B‖ / ‖Δ_AB‖` | ratio | →1 | §7 measured **1.13** — the sum overshoots |
| **shuffled floor** | composed cosine with **both** deltas permuted across episodes | −1…1 | — | the chance level. Permuting only one is **not** a floor |
| **observation ceiling** | `‖AB − (A + B − base)‖ / ‖AB − base‖` on the **rendered observations** | ratio | ↓ | the two objects share rays, so the render is not additive either. **No latent that is a faithful function of the observation can have a smaller residual than this.** It is the reference scale every residual below is read against |

**Displacement scale** `s` rescales each target as `pos[ef] + s·(target − pos[ef])`, so `s = 1` is
the real teleport and `s → 0` is the limit where first-order Taylor behaviour must dominate.

**Models** — all `H256` GRUs on `4_fixed_refl_inview`, rows from `NONLINEAR_GRU_RUNS.md`:

| code | label | enc / dec depth | decoder | why |
|---|---|---|---|---|
| `runs/controls/H256` | **linear enc+dec** | 0 / 0 | affine | **the baseline** — the exact checkpoint the editability findings and `delta_h_analysis` were established on |
| `runs/nonlinear_gru/NL_enc2dec2_s0` | **nonlinear enc+dec** | 2 / 2 | nonlinear | the variant that breaks the affine-decoder identity |

Each is run **trained** and **randomly initialised at the identical config**, two seeds.


In [ ]:
# [1] Setup. Every recurring computation lives in `composition_lib.py` beside this notebook;
#     the notebook is the orchestrator (CLAUDE.md invariant 4).
import json, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

THREAD = Path.cwd()
REPO = THREAD.parents[3]
for p in (str(THREAD), str(REPO), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
import composition_lib as cl
from pim.figures.theme import PALETTE, style_ax

FIGDIR = REPO / "runs" / "latent_linearity" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

N = 256                       # episodes from edits.h5
SCALES = (1.0, 0.5, 0.25, 0.125)
SEED = 0
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# §7's published numbers, cited not recomputed (delta_h_analysis, 2026-08-03, GRU counterfactual)
DH7 = {"cos": 0.873, "resid_lo": 0.39, "resid_hi": 0.69, "norm_ratio": 1.13, "shuffled": 0.059}

torch.manual_seed(SEED)
np.random.seed(SEED)
RESULTS = {}
print(f"device {DEV} · N = {N} episodes · displacement scales {SCALES}")
print(f"reference (delta_h_analysis §7, GRU counterfactual): cos {DH7['cos']:+.3f}, "
      f"resid {DH7['resid_lo']}–{DH7['resid_hi']}, ‖comp‖/‖direct‖ {DH7['norm_ratio']}")


In [ ]:
# [2] The edit set — real teleports for BOTH objects, and correctness gates on it.
sim, cfg, rad, refl = cl.world()
pos, vel, uned, tgt = cl.edit_set(N, seed=SEED)

# ⛔ Displacement is measured from the UN-TELEPORTED world, not from positions[ef]. On edits.h5
#    the teleport is already in the data, so measuring from positions[ef] gives ZERO for each
#    episode's own edit object — which silently made half the single-object edits null in a first
#    pass on 2026-08-21.
disp = np.linalg.norm(tgt - uned, axis=-1)
sep = np.linalg.norm(tgt[:, 0] - tgt[:, 1], axis=-1)
assert disp.min() > 1e-3, "some 'teleport' does not move the object"
assert np.isfinite(tgt).all() and np.isfinite(uned).all()
assert np.abs(uned - pos[:, cl.EF]).max() > 1e-3, "un-teleported world is identical to the data"

print(f"{N} episodes from edits.h5, edit frame {cl.EF}")
print(f"  displacement per object : mean {disp.mean():.2f}  p10 {np.percentile(disp, 10):.2f}  "
      f"p90 {np.percentile(disp, 90):.2f} sim units")
print(f"  separation of the two targets: mean {sep.mean():.2f}  min {sep.min():.2f}")
print(f"  position std in the data      : {pos[:, cl.EF].reshape(-1, 2).std(0).mean():.2f} "
      f"(the scale these displacements are read against)")
print(f"  |uned - positions[ef]| mean   : {np.abs(uned - pos[:, cl.EF]).mean():.3f} "
      f"(the teleport that is already in the data)")
RESULTS["displacement"] = {"mean": float(disp.mean()), "p10": float(np.percentile(disp, 10)),
                           "p90": float(np.percentile(disp, 90))}

In [ ]:
# [3] Render the four counterfactual worlds at every displacement scale, and measure the
#     OBSERVATION's own non-additivity — the floor no latent can beat. ~2 min.
t0 = time.time()
HIST, CEIL = {}, {}
for s in SCALES:
    HIST[s] = cl.histories(pos, vel, uned, tgt, s, sim, cfg, rad, refl)
    CEIL[s] = cl.render_ceiling(HIST[s])
    d = HIST[s]
    print(f"  scale {s:<6} mean |A−base| {np.abs(d['A']-d['base']).mean():.4f}   "
          f"|AB−base| {np.abs(d['AB']-d['base']).mean():.4f}   "
          f"observation ceiling {CEIL[s]:.3f}", flush=True)
print(f"\nrendered in {time.time()-t0:.0f}s")
print("The ceiling is nonzero because the two objects SHARE RAYS — the render itself is not")
print("additive. Every latent residual below is read against the ceiling at its own scale.")
RESULTS["ceiling"] = {str(k): v for k, v in CEIL.items()}

In [ ]:
# [4] Every model at every scale. Trained and random-init at the identical config, two seeds.
t0 = time.time()
MODELS = cl.models(device=DEV, seeds=(0, 1))
M = {}
for s in SCALES:
    for tag, m in MODELS.items():
        M[(tag, s)] = cl.metrics(cl.latents(m, HIST[s], device=DEV), seed=SEED)
print(f"{len(M)} model x scale cells in {time.time()-t0:.0f}s\n")

ORDER = list(MODELS)
RESULTS["metrics"] = {f"{t}|{s}": v for (t, s), v in M.items()}


def table(scale):
    rows = ["| model | composed cos ↑ | *triviality* `cos(Δ_A,Δ_AB)` | relative residual ↓ | "
            "*residual, one delta* | ‖comp‖/‖direct‖ | shuffled floor |", "|---|---|---|---|---|---|---|",
            f"| *observation ceiling* | — | — | **{CEIL[scale]:.3f}** | — | — | — |"]
    for t in ORDER:
        r = M[(t, scale)]
        rows.append(f"| {t} | {r['cos']:+.3f} | *{r['cos_trivial']:+.3f}* | **{r['resid']:.3f}** | "
                    f"*{r['resid_trivial']:.3f}* | {r['norm_ratio']:.3f} | {r['cos_shuffled']:+.3f} |")
    return "\n".join(rows)


display(Markdown(f"**Table 1 — real teleports (displacement scale 1.0), N = {N}.** "
                 f"Italic columns are baselines, not targets.\n\n" + table(1.0)))
print(f"delta_h_analysis §7 reference: cos {DH7['cos']:+.3f} · resid {DH7['resid_lo']}–{DH7['resid_hi']} "
      f"· ‖comp‖/‖direct‖ {DH7['norm_ratio']} · shuffled {DH7['shuffled']:+.3f}")


In [ ]:
# [5] Table 2 — the magnitude sweep, which is the actual diagnostic. If composition were purely
#     first-order Taylor behaviour, every model would converge to the observation ceiling as the
#     displacement shrinks, and trained and random would become indistinguishable.
rows = ["| model | " + " | ".join(f"s = {s}" for s in SCALES) + " |", "|---|" + "---|" * len(SCALES)]
rows.append("| *observation ceiling* | " + " | ".join(f"**{CEIL[s]:.3f}**" for s in SCALES) + " |")
for t in ORDER:
    rows.append(f"| {t} | " + " | ".join(f"{M[(t, s)]['resid']:.3f}" for s in SCALES) + " |")
display(Markdown("**Table 2 — relative residual by displacement scale (lower is better).**\n\n"
                 + "\n".join(rows)))

rows2 = ["| model | " + " | ".join(f"s = {s}" for s in SCALES) + " |", "|---|" + "---|" * len(SCALES)]
for t in ORDER:
    rows2.append(f"| {t} | " + " | ".join(f"{M[(t, s)]['cos']:+.3f}" for s in SCALES) + " |")
display(Markdown("**Table 3 — composed cosine by displacement scale.**\n\n" + "\n".join(rows2)))

print("EXCESS over the observation ceiling — the non-additivity the MODEL adds, which is the only")
print("part attributable to the network rather than to the renderer:")
print(f"{'model':<32} " + " ".join(f"{'s=' + str(s):>9}" for s in SCALES))
for t in ORDER:
    print(f"{t:<32} " + " ".join(f"{M[(t, s)]['resid'] - CEIL[s]:>9.3f}" for s in SCALES))
RESULTS["excess"] = {f"{t}|{s}": M[(t, s)]["resid"] - CEIL[s] for t in ORDER for s in SCALES}


In [ ]:
# [6] Fig 1 — the whole result. Absolute quantities, never gains; every baseline drawn on the same
#     axis as the thing it bounds.
STYLE = {}
for t in ORDER:
    trained = t.startswith("TRAINED")
    lin = "nonlinear" not in t
    STYLE[t] = dict(color=PALETTE[0] if lin else PALETTE[3],
                    ls="-" if trained else "--",
                    marker="o" if trained else "^",
                    lw=2.2 if trained else 1.6,
                    alpha=1.0 if trained else 0.85)
x = np.array(SCALES)
fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.0), facecolor="white")

for t in ORDER:
    axes[0].plot(x, [M[(t, s)]["resid"] for s in SCALES], label=t, ms=6, **STYLE[t])
axes[0].plot(x, [CEIL[s] for s in SCALES], color="0.25", ls=":", lw=2.4, marker="s", ms=6,
             label="observation ceiling (the renderer's own non-additivity)")
axes[0].axhspan(DH7["resid_lo"], DH7["resid_hi"], color="0.85", zorder=0,
                label=f"delta_h_analysis §7 reported range ({DH7['resid_lo']}–{DH7['resid_hi']})")
axes[0].set_ylabel("relative residual (lower is better)")
axes[0].set_title("(a) does composition explain the double edit?")
axes[0].set_ylim(0, 0.8)

for t in ORDER:
    axes[1].plot(x, [M[(t, s)]["resid"] - CEIL[s] for s in SCALES], label=t, ms=6, **STYLE[t])
axes[1].axhline(0.0, color="0.25", ls=":", lw=2.4, label="the renderer's limit (excess = 0)")
axes[1].set_ylabel("excess residual over the ceiling")
axes[1].set_title("(b) the non-additivity the MODEL adds")

for t in ORDER:
    axes[2].plot(x, [M[(t, s)]["cos"] for s in SCALES], label=t, ms=6, **STYLE[t])
    st = dict(STYLE[t]); st["alpha"] = 0.35; st["lw"] = 1.1; st["marker"] = "."
    axes[2].plot(x, [M[(t, s)]["cos_trivial"] for s in SCALES], **st)
axes[2].plot([], [], color="0.5", lw=1.1, marker=".", alpha=0.5,
             label="faint = triviality baseline cos(Δ_A, Δ_AB)")
axes[2].axhline(DH7["cos"], color="0.45", ls="--", lw=1.8, label=f"§7 reported ({DH7['cos']:+.3f})")
axes[2].axhline(np.mean([M[(t, 1.0)]["cos_shuffled"] for t in ORDER]), color="0.7", ls="-.", lw=1.6,
                label="shuffled floor (both deltas permuted)")
axes[2].set_ylabel("composed cosine")
axes[2].set_title("(c) direction only — scale-blind, and the weaker readout")
axes[2].set_ylim(-0.05, 1.02)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([str(s) for s in SCALES])
    ax.minorticks_off()          # log minor ticks collide with the explicit scale labels
    ax.set_xlabel("displacement scale (1.0 = the real teleport in edits.h5)")
    ax.legend(fontsize=7, handlelength=2.8)
    style_ax(ax)
fig.suptitle("Fig 1 — latent object-composition: trained vs randomly initialised, "
             f"by displacement magnitude (N = {N})", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94), w_pad=2.0)
fig.savefig(FIGDIR / "fig1_composition_random_baseline.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# [7] Serialise everything, curves included (`harness/ANALYSIS.md` §1 — a filter that drops lists
#     on the way to JSON has silently destroyed required curves in this repo before).
out = REPO / "runs" / "latent_linearity" / "random_baseline_results.json"
RESULTS["config"] = {"n_episodes": N, "scales": list(SCALES), "seed": SEED,
                     "edit_frame": cl.EF, "device": DEV,
                     "checkpoints": cl.CKPT, "arch": {k: dict(v) for k, v in cl.ARCH.items()}}
RESULTS["delta_h7_reference"] = DH7
out.write_text(json.dumps(RESULTS, indent=1, default=float))

lin_t = [M[("TRAINED linear enc+dec", s)]["resid"] - CEIL[s] for s in SCALES]
lin_r = [np.mean([M[(f"RANDOM s{k} linear enc+dec", s)]["resid"] for k in (0, 1)]) - CEIL[s]
         for s in SCALES]
nl_t = [M[("TRAINED nonlinear enc+dec", s)]["resid"] - CEIL[s] for s in SCALES]
nl_r = [np.mean([M[(f"RANDOM s{k} nonlinear enc+dec", s)]["resid"] for k in (0, 1)]) - CEIL[s]
        for s in SCALES]
print(f"wrote {out}\n")
print("HEADLINE NUMBERS (excess residual over the observation ceiling; 0 = the renderer's limit)")
print(f"  {'scale':>8} {'lin trained':>12} {'lin random':>11} {'nonlin trained':>15} {'nonlin random':>14}")
for i, s in enumerate(SCALES):
    print(f"  {s:>8} {lin_t[i]:>12.3f} {lin_r[i]:>11.3f} {nl_t[i]:>15.3f} {nl_r[i]:>14.3f}")
print(f"\nfigure: {FIGDIR / 'fig1_composition_random_baseline.png'}")


## Summary

### Current results (updated 2026-08-21)

**The claim survives, but only in a restated and much weaker form. The strong version — "the latent
superposes object edits, and that is a learned property" — is not supported by the cosine, which is
what §7 led with.**

**1. By cosine, composition is largely architectural.** At the real teleport scale, trained
linear reads **+0.904** and a randomly initialised network of the identical config reads **+0.890**.
On the nonlinear model the random net is *higher* than the trained one (+0.853 vs +0.835). Training
moves the composed cosine by 0.014 on one architecture and −0.018 on the other.

The cosine is anchored, so this is not a measurement failure: trained linear reproduces §7's
**+0.873** and `‖composed‖/‖direct‖` reproduces its **1.13**, and the corrected shuffled floor
(**both** deltas permuted) lands at **+0.026** against §7's +0.059.

**2. Composition is doing real work — the metric is not vacuous.** The triviality baseline
`cos(Δ_A, Δ_AB)` sits at **+0.40 … +0.58**, far below the composed cosine, and the one-delta
residual is 0.72–0.92 against a composed residual of 0.21–0.65. Adding the second delta genuinely
improves both direction and magnitude. So "composition explains the double edit" is true; what is
*not* established is that training is why.

**3. Read against the renderer's own limit, training does separate — on the linear model.** The
observation is itself non-additive because the two objects share rays, and no latent that is a
faithful function of it can do better. The excess residual **over that ceiling** is the only part
attributable to the network:

| displacement scale | 1.0 | 0.5 | 0.25 | 0.125 |
|---|---|---|---|---|
| observation ceiling | 0.406 | 0.373 | 0.285 | 0.207 |
| **trained linear** | **+0.046** | **−0.004** | **−0.019** | **+0.004** |
| random linear (mean of 2 seeds) | +0.104 | +0.064 | +0.037 | +0.028 |
| trained nonlinear | +0.241 | +0.138 | +0.077 | +0.073 |
| random nonlinear (mean of 2 seeds) | +0.208 | +0.150 | +0.121 | +0.109 |

**The trained linear model composes as well as its own input allows — excess ≈ 0 at every scale.**
Random initialisations of the identical architecture add 0.03–0.10 of their own non-additivity.
That gap is consistent across seeds and across the whole magnitude sweep, and it is the defensible
form of the claim.

**4. The nonlinear model shows no training benefit, and is worse than its own random baseline at
full teleport scale** (+0.241 vs +0.208). Whatever the affine model learned about keeping objects
separable, the nonlinear one did not.

**5. The magnitude sweep confirms the Taylor mechanism is a large part of it.** Every model's
residual falls as the displacement shrinks, and so does the ceiling. Additivity is a first-order
property of any smooth map; the trained-vs-random gap persists across the sweep but narrows
(0.058 → 0.024 on the linear model), which is what "part mechanism, part learned" looks like.

### What this does and does not license

**Does:** "the trained affine-decoder GRU's configuration→latent map introduces essentially no
non-additivity of its own beyond what the renderer forces" — a sharper claim than §7's, with a
reference scale.

**Does not:** any claim resting on the composed cosine alone. A random network reproduces it.

⚠ **And it cannot speak to §7's decisive readout at all.** §7 measured the composed state
*applied and rolled out*, at **83–87% of the direct edit's Edit-Index gain**, warning in those words
that *"vector agreement alone can mislead"*. A randomly initialised decoder emits garbage, so that
readout is undefined for the random arms. **Everything here is a vector metric**, i.e. the weaker
readout §7 already cautioned against — which is a limit of what any random baseline can establish
here, not an oversight.

### Provenance of the corrections in this notebook

A first pass on 2026-08-21 got three things wrong, all of which inflated the apparent result and
all of which are fixed here: it displaced every object in the same direction (making Δ_A ∥ Δ_B), it
shuffled only one delta when computing the floor (reporting +0.38, which is bounded below by the
triviality baseline and is therefore not a floor), and it measured displacement from
`positions[ef]` on `edits.h5` — where the teleport is **already in the data**, so each episode's own
edit object did not move at all.